# 结构化输出 Structured Output
> 让 LLM 返回符合预定义 schema 的结构化数据（Pydantic 模型 / TypedDict / JSON Schema），而不是自由文本。

LangChain 提供两种主要方式：
1. `with_structured_output(schema)` —— 把 LLM 包装成「输出符合 schema 的可调用对象」，最常用
2. `bind_tools(tools)` —— 通过工具调用间接得到结构化参数

结构化输出底层依赖模型自身的 function-calling / JSON-mode 能力，因此**模型必须支持工具调用**。

## 0. 准备 LLM
统一初始化一个支持工具调用的 ChatOpenAI，后续所有示例复用。

In [3]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-4o-mini"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)
print("LLM 就绪：", llm.model_name)

LLM 就绪： mimo-v2.5-pro


## 1. 使用 Pydantic 模型定义输出结构（推荐）
用 Pydantic `BaseModel` 描述字段、类型、描述、约束。`with_structured_output` 会把 schema 转成 function-calling 工具传给模型，返回值是**该 Pydantic 模型的实例**，可直接用属性访问。

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


class BookReview(BaseModel):
    """对一本书的评价。"""
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    rating: int = Field(ge=1, le=5, description="评分，1-5 分")
    sentiment: Literal["正面", "中性", "负面"] = Field(description="整体情感倾向")
    summary: str = Field(description="一句话点评")
    keywords: list[str] = Field(default_factory=list, description="关键词列表")


# 包装 LLM：之后 invoke 返回的就是 BookReview 实例
structured_llm = llm.with_structured_output(BookReview)

review = structured_llm.invoke("请评价《深入理解计算机系统》这本书，作者是 Randal E. Bryant。")

print("类型：", type(review))
print("书名：", review.title)
print("作者：", review.author)
print("评分：", review.rating)
print("情感：", review.sentiment)
print("点评：", review.summary)
print("关键词：", review.keywords)

# 因为是 Pydantic 实例，可以直接序列化成 dict / JSON
print("\nJSON：", review.model_dump_json(indent=2))

ValidationError: 1 validation error for BookReview
  Invalid JSON: key must be a string at line 1 column 2 [type=json_invalid, input_value='{The user is asking me t...⭐⭐⭐⭐（5/5）**', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

## 2. 使用 TypedDict 定义输出结构（轻量级）
不需要校验逻辑、只想要一个带类型标注的字典时，可用 `TypedDict`。返回值是普通 `dict`，但 IDE 有类型提示。

In [ ]:
from typing import TypedDict, Literal


class WeatherInfo(TypedDict):
    """天气信息。"""
    city: str
    temperature: int          # 气温（摄氏度）
    condition: Literal["晴", "多云", "雨", "雪"]
    suggestion: str           # 出行建议


weather_llm = llm.with_structured_output(WeatherInfo)

info = weather_llm.invoke("假设今天上海气温 22 度、多云，给我一份天气简报。")

print("类型：", type(info))   # <class 'dict'>
print("城市：", info["city"])
print("气温：", info["temperature"])
print("天气：", info["condition"])
print("建议：", info["suggestion"])

## 3. 使用 JSON Schema（dict）定义输出结构
不想引入 Pydantic / TypedDict，也可以直接传一个 JSON Schema 字典。适合动态生成 schema 的场景。

In [ ]:
person_schema = {
    "title": "Person",
    "description": "从一段自然语言中提取的人物信息。",
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "姓名"},
        "age": {"type": "integer", "description": "年龄"},
        "hobbies": {
            "type": "array",
            "items": {"type": "string"},
            "description": "爱好列表",
        },
    },
    "required": ["name", "age"],
}

person_llm = llm.with_structured_output(person_schema)

person = person_llm.invoke("张三今年 28 岁，平时喜欢爬山、摄影和写代码。")

print("类型：", type(person))
print("姓名：", person["name"])
print("年龄：", person["age"])
print("爱好：", person.get("hobbies"))

## 4. 嵌套结构与列表
Pydantic 模型可以嵌套、可以包含列表，适合抽取复杂关系数据。下面示例抽取一篇技术文章的结构化大纲。

In [ ]:
from pydantic import BaseModel, Field


class Section(BaseModel):
    """文章中的一个章节。"""
    heading: str = Field(description="章节标题")
    bullet_points: list[str] = Field(description="该章节的要点列表")


class ArticleOutline(BaseModel):
    """技术文章的结构化大纲。"""
    title: str = Field(description="文章标题")
    topic: str = Field(description="文章主题")
    sections: list[Section] = Field(description="章节列表")
    estimated_reading_minutes: int = Field(ge=1, description="预计阅读时长（分钟）")


outline_llm = llm.with_structured_output(ArticleOutline)

text = (
    "《LangChain 入门》主要介绍 LangChain 的核心概念，包含三部分："
    "第一部分讲 Models 与 Prompts，要点是 ChatModel 调用方式和 PromptTemplate 模板复用；"
    "第二部分讲 Chains，要点是 LCEL 表达式和链式组合；"
    "第三部分讲 Agents 与 Tools，要点是工具绑定和自动调用。整篇文章读完大约 12 分钟。"
)

outline = outline_llm.invoke(text)

print(f"标题：{outline.title}")
print(f"主题：{outline.topic}")
print(f"预计阅读：{outline.estimated_reading_minutes} 分钟")
print("章节：")
for i, sec in enumerate(outline.sections, 1):
    print(f"  {i}. {sec.heading}")
    for p in sec.bullet_points:
        print(f"     - {p}")

## 5. method 参数：function_calling vs json_mode
`with_structured_output` 的 `method` 参数控制底层机制：
- `"function_calling"`（默认）：用 function-calling，依赖模型工具调用能力，**字段描述最准确**
- `"json_mode"`：用 OpenAI JSON mode，把 schema 塞进 `response_format`，**不依赖工具调用但约束较弱**

当模型不支持 function-calling 但支持 JSON 输出时，可用 `json_mode`。

In [ ]:
from pydantic import BaseModel, Field


class CityFact(BaseModel):
    """城市基本事实。"""
    city: str = Field(description="城市名")
    country: str = Field(description="所在国家")
    population_millions: float = Field(description="人口（百万）")
    is_capital: bool = Field(description="是否首都")


# 方式一：function_calling（默认）
fact_fc = llm.with_structured_output(CityFact, method="function_calling").invoke("介绍一下北京。")
print("[function_calling]", fact_fc.model_dump())

# 方式二：json_mode
fact_json = llm.with_structured_output(CityFact, method="json_mode").invoke("介绍一下东京。")
print("[json_mode]      ", fact_json.model_dump())

## 6. 在 Chain 中组合结构化输出（LCEL）
`with_structured_output` 返回的是一个 `Runnable`，可以和 prompt、parser 等组合进 LCEL 链。

下面用「prompt → 结构化 LLM」组成一条抽取链，并演示 `batch` 批量抽取。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ProductInfo(BaseModel):
    """从商品描述中抽取的结构化信息。"""
    name: str = Field(description="商品名称")
    brand: str = Field(description="品牌")
    price: float = Field(ge=0, description="价格（元）")
    in_stock: bool = Field(description="是否有货")


prompt = ChatPromptTemplate.from_template(
    "请从下面这段商品描述中抽取结构化信息：\n\n{description}"
)

# LCEL 链：prompt | structured_llm
chain = prompt | llm.with_structured_output(ProductInfo)

descriptions = [
    "苹果 Apple iPhone 16 Pro，售价 8999 元，目前现货充足。",
    "华为 MatePad Pro 13.2 英寸平板，售价 4999 元，暂时缺货。",
]

results = chain.batch([{"description": d} for d in descriptions])

for r in results:
    print(r.model_dump())

## 7. 错误处理与重试
模型偶尔会输出不符合 schema 的内容。可以：
- 用 `include_raw=True` 同时拿到原始 AIMessage，便于排查
- 在外层用 `with_retry()` 自动重试

In [ ]:
from pydantic import BaseModel, Field


class OrderItem(BaseModel):
    """订单中的一项商品。"""
    sku: str = Field(description="商品 SKU 编号")
    quantity: int = Field(ge=1, description="购买数量")
    unit_price: float = Field(ge=0, description="单价")


# include_raw=True 返回 {'raw': AIMessage, 'parsed': Model|None, 'parsing_error': Exception|None}
robust_llm = llm.with_structured_output(OrderItem, include_raw=True).with_retry(stop_after_attempt=2)

res = robust_llm.invoke("我买了一瓶可乐，SKU 是 COLA-330ML，3 瓶，每瓶 3.5 元。")

if res["parsing_error"] is None:
    print("解析成功：", res["parsed"].model_dump())
else:
    print("解析失败：", res["parsing_error"])
    print("原始输出：", res["raw"].content)

## 8. 用 bind_tools 间接实现结构化输出
把 Pydantic 模型当作「工具」传给 `bind_tools`，模型调用该工具时返回的 `tool_calls` 就是结构化参数。比 `with_structured_output` 更底层、更灵活（可以同时绑定多个工具，让模型自己选）。

In [ ]:
from pydantic import BaseModel, Field


class ExtractPerson(BaseModel):
    """从文本中抽取的人物信息。"""
    name: str = Field(description="姓名")
    age: int | None = Field(default=None, description="年龄，未知则不填")
    occupation: str | None = Field(default=None, description="职业，未知则不填")


llm_with_tool = llm.bind_tools([ExtractPerson])

response = llm_with_tool.invoke("李雷是一名 30 岁的软件工程师。")

if response.tool_calls:
    call = response.tool_calls[0]
    print("模型选择调用工具：", call["name"])
    print("结构化参数：", call["args"])
    # 用 Pydantic 模型再做一次校验
    person = ExtractPerson.model_validate(call["args"])
    print("校验后的对象：", person.model_dump())
else:
    print("模型未调用工具，原始输出：", response.content)

## 小结

| 方式 | 返回类型 | 适用场景 |
|---|---|---|
| `with_structured_output(Pydantic)` | Pydantic 实例 | 需要**校验**、字段约束、嵌套结构（最常用） |
| `with_structured_output(TypedDict)` | dict | 只需类型标注，不需要运行时校验 |
| `with_structured_output(dict_schema)` | dict | 动态生成 schema，不想引入 Pydantic |
| `with_structured_output(..., method="json_mode")` | 同上 | 模型不支持 function-calling 时 |
| `with_structured_output(..., include_raw=True)` | dict(raw/parsed/parsing_error) | 需要容错与排查 |
| `bind_tools([Model])` | AIMessage.tool_calls | 需要更底层控制、多工具并存 |

**关键点**：
- 结构化输出依赖模型的 function-calling / JSON-mode 能力，选模型时要注意
- Pydantic 字段的 `description` 会被转成 schema 描述发给模型，**写好 description 是输出质量的关键**
- 复杂嵌套、列表、枚举（`Literal`）、范围约束（`Field(ge=, le=)`）都支持
- 结构化输出本身是 `Runnable`，可无缝接入 LCEL 链、`batch`、`stream` 等接口